# HADO 대회 영상 압축 파이프라인

Google Drive에 있는 원본 영상(80초)을 YOLOv8 추적에 적합한 크기로 압축합니다.

| 항목 | 원본 | 압축 후 |
|------|------|---------|
| 해상도 | 1080p | 720p |
| FPS | 30 | 24 |
| 코덱 | 다양 | H.264 (CRF 26) |
| 예상 용량 | 300~500 MB | 40~70 MB |

**실행 순서**: 셀을 위에서 아래로 순서대로 실행하세요.

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive 마운트 완료')

## 2. 설정 — 폴더 경로만 본인 환경에 맞게 수정하세요

In [ ]:
from pathlib import Path

# ── 여기만 수정 ──────────────────────────────────────────
INPUT_DIR  = Path('/content/drive/MyDrive/HADO_videos/raw')       # 원본 영상 폴더
OUTPUT_DIR = Path('/content/drive/MyDrive/HADO_videos/compressed') # 압축 결과 폴더
# ────────────────────────────────────────────────────────

# 압축 설정
TARGET_WIDTH  = 1280
TARGET_HEIGHT = 720
TARGET_FPS    = 24
CRF           = 26   # 18=고화질 / 26=균형 / 32=소용량

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 원본 영상 목록
videos = sorted(INPUT_DIR.glob('*.mp4')) + sorted(INPUT_DIR.glob('*.MP4'))
print(f'원본 영상: {len(videos)}개')
for v in videos[:5]:
    print(f'  {v.name}')
if len(videos) > 5:
    print(f'  ... 외 {len(videos)-5}개')

## 3. ffmpeg 확인

In [ ]:
import subprocess
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print(result.stdout.split('\n')[0])
print('ffmpeg 사용 가능 ✓')

## 4. 단일 영상 테스트 (전체 실행 전 먼저 확인)

In [ ]:
import os

def compress_video(src: Path, dst: Path) -> dict:
    """영상 1개를 압축. 결과 통계 dict 반환."""
    cmd = [
        'ffmpeg', '-y',
        '-i', str(src),
        '-vf', f'scale={TARGET_WIDTH}:{TARGET_HEIGHT}',
        '-r', str(TARGET_FPS),
        '-c:v', 'libx264',
        '-crf', str(CRF),
        '-preset', 'fast',
        '-c:a', 'aac', '-b:a', '64k',
        str(dst)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr[-500:])

    src_mb = src.stat().st_size / 1e6
    dst_mb = dst.stat().st_size / 1e6
    return {'src_mb': src_mb, 'dst_mb': dst_mb,
            'ratio': dst_mb / src_mb * 100}

# 첫 번째 영상으로 테스트
test_src = videos[0]
test_dst = OUTPUT_DIR / test_src.name

print(f'테스트: {test_src.name}')
stats = compress_video(test_src, test_dst)
print(f'  원본:  {stats["src_mb"]:.1f} MB')
print(f'  압축:  {stats["dst_mb"]:.1f} MB ({stats["ratio"]:.0f}%)')

## 5. 전체 영상 일괄 압축

In [ ]:
from tqdm.notebook import tqdm
import time

results = []
skipped = 0
failed  = []

for src in tqdm(videos, desc='압축 진행'):
    dst = OUTPUT_DIR / src.name

    # 이미 처리된 파일은 건너뜀 (재실행 안전)
    if dst.exists() and dst.stat().st_size > 1e6:
        skipped += 1
        continue

    try:
        stats = compress_video(src, dst)
        stats['name'] = src.name
        results.append(stats)
    except Exception as e:
        print(f'\n[실패] {src.name}: {e}')
        failed.append(src.name)

# 요약
print('\n' + '='*50)
print(f' 완료: {len(results)}개  |  건너뜀: {skipped}개  |  실패: {len(failed)}개')
if results:
    total_src = sum(r['src_mb'] for r in results)
    total_dst = sum(r['dst_mb'] for r in results)
    print(f' 원본 합계: {total_src/1024:.1f} GB')
    print(f' 압축 합계: {total_dst/1024:.1f} GB  ({total_dst/total_src*100:.0f}%)')
if failed:
    print(f'\n실패 목록:')
    for f in failed:
        print(f'  {f}')

## 6. 압축 결과 확인

In [ ]:
import cv2

# 첫 번째 압축 영상 메타데이터 확인
sample = next(OUTPUT_DIR.glob('*.mp4'))
cap = cv2.VideoCapture(str(sample))

print(f'파일: {sample.name}')
print(f'해상도: {int(cap.get(3))}×{int(cap.get(4))} px')
print(f'FPS: {cap.get(5):.1f}')
print(f'프레임 수: {int(cap.get(7))}')
print(f'길이: {cap.get(7)/cap.get(5):.1f}초')
print(f'용량: {sample.stat().st_size/1e6:.1f} MB')
cap.release()

# 전체 압축 파일 수 확인
compressed = list(OUTPUT_DIR.glob('*.mp4'))
total_size  = sum(f.stat().st_size for f in compressed) / 1e9
print(f'\n압축 완료 파일: {len(compressed)}개  |  총 {total_size:.1f} GB')